# Homework 13

## Импорты, seed и среда

In [6]:

# Базовые библиотеки для воспроизводимости, анализа и удобного отображения результатов.
import random
from typing import Iterable

import numpy as np
import pandas as pd
import torch

from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from collections import Counter

In [7]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)

print(f"Device: {device}")

Device: mps


In [8]:
# Берём стандартную мультиязычную BERT-модель: она хорошо подходит для русскоязычных примеров.
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)

Tokenizer loaded: bert-base-multilingual-cased
Tokenizer class: BertTokenizer
Model max length: 512


## Датасет и первичный анализ

In [9]:
dataset = load_dataset("dair-ai/emotion")

Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 770799.23 examples/s]


In [10]:
for split_name, split_data in dataset.items():
    print(f"{split_name:12s}: {len(split_data):>6} примеров")

train       :  16000 примеров
validation  :   2000 примеров
test        :   2000 примеров


In [14]:
label_names = dataset["train"].features["label"].names
print("Классы:", label_names)

Классы: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [12]:
train_labels = dataset["train"]["label"]
counts = Counter(train_labels)
print("\nРаспределение классов (train):")
for idx, count in sorted(counts.items()):
    print(f"  {label_names[idx]:10s}: {count:>5} ({count/len(train_labels)*100:.1f}%)")


Распределение классов (train):
  sadness   :  4666 (29.2%)
  joy       :  5362 (33.5%)
  love      :  1304 (8.2%)
  anger     :  2159 (13.5%)
  fear      :  1937 (12.1%)
  surprise  :   572 (3.6%)


In [13]:
print("\n 5 примеров из train ")
for i in range(5):
    ex = dataset["train"][i]
    print(f"  [{label_names[ex['label']]:8s}] {ex['text'][:80]}")

print("\n 3 примера из validation ")
for i in range(3):
    ex = dataset["validation"][i]
    print(f"  [{label_names[ex['label']]:8s}] {ex['text'][:80]}")


 5 примеров из train 
  [sadness ] i didnt feel humiliated
  [sadness ] i can go from feeling so hopeless to so damned hopeful just from being around so
  [anger   ] im grabbing a minute to post i feel greedy wrong
  [love    ] i am ever feeling nostalgic about the fireplace i will know that it is still on 
  [anger   ] i am feeling grouchy

 3 примера из validation 
  [sadness ] im feeling quite sad and sorry for myself but ill snap out of it soon
  [sadness ] i feel like i am still looking at a blank canvas blank pieces of paper
  [love    ] i feel like a faithful servant


В датасете классифицируется эмоциональная окраска твита, насторение автора.

## Токенизация

In [ ]:
def inspect_single_text(text: str, tokenizer) -> pd.DataFrame:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    decoded_pieces = [tokenizer.decode([tid]) for tid in token_ids]
    return pd.DataFrame(
        {
            "position": list(range(len(tokens))),
            "token": tokens,
            "token_id": token_ids,
            "decoded_piece": decoded_pieces,
        }
    )

example_text = dataset['train'][1]['text']
print("Исходный текст:")
print(example_text)
print()

Исходный текст:
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake



In [20]:
tokens = tokenizer.tokenize(example_text)
token_ids = tokenizer.encode(example_text, add_special_tokens=True)
decoded_text = tokenizer.decode(token_ids)

print("Токены:")
print(tokens)
print()

print("ID токенов:")
print(token_ids)
print()

print("decode(token_ids):")
print(decoded_text)

Токены:
['i', 'can', 'go', 'from', 'feeling', 'so', 'hope', '##less', 'to', 'so', 'dam', '##ned', 'hope', '##ful', 'just', 'from', 'being', 'around', 'someone', 'who', 'care', '##s', 'and', 'is', 'aw', '##ake']

ID токенов:
[101, 177, 10944, 11783, 10188, 61362, 10380, 50725, 14985, 10114, 10380, 39121, 17021, 50725, 14446, 12820, 10188, 11223, 12166, 30455, 10479, 11131, 10107, 10111, 10124, 56237, 26389, 102]

decode(token_ids):
[CLS] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake [SEP]
